# 🎙️ ROTBOTS — The Voice
## Text-to-Speech Narration

---

This notebook generates **spoken narration** for each scene using Edge-TTS (Microsoft's free text-to-speech).

1. Load the essay script from Google Drive
2. Choose a voice
3. Generate audio for each scene
4. Preview and save

> **🤔 Think about:** What does a synthetic voice do to trust? Would you believe this narration more or less than a human voice?

---

## 🔧 Setup

In [ ]:
# ============================================================
# SETUP
# ============================================================

!pip install -q edge-tts

import json
import asyncio
import subprocess
from pathlib import Path
from IPython.display import display, Markdown, Audio

# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
WORK_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = WORK_DIR / 'audio'
AUDIO_DIR.mkdir(exist_ok=True)

print('✅ Edge-TTS installed')
print(f'📁 Workspace: {WORK_DIR}')
print(f'🎙️ Audio folder: {AUDIO_DIR}')

In [ ]:
# ============================================================
# LOAD ESSAY SCRIPT from Google Drive
# ============================================================

essay_file = WORK_DIR / 'essay_script.json'

if essay_file.exists():
    with open(essay_file) as f:
        essay = json.load(f)
    print(f'✅ Loaded essay: {essay.get("title", "Untitled")}')

    # Extract narrations from all sections
    narrations = []
    for chapter in essay.get('chapters', []):
        for section in chapter.get('sections', []):
            narration_text = section.get('narration', '')
            if narration_text:
                narrations.append({
                    'chapter': chapter.get('chapter', 0),
                    'section': section.get('section', 0),
                    'narration': narration_text,
                    'mood': section.get('mood', '')
                })
    print(f'   Found {len(narrations)} narration sections')
    for n in narrations:
        print(f'   Ch{n["chapter"]}.{n["section"]}: {n["narration"][:60]}...')
else:
    print('❌ No essay_script.json found!')
    print('   Run the Script Writer notebook first.')

---
## 🎙️ Station 4: Choose a Voice

Edge-TTS offers many natural-sounding voices for free. Try different ones!

> **🤔 Think about:** How does the voice change the feeling of the essay? A male voice vs. female? British vs. American?

In [ ]:
# ============================================================
# VOICE SELECTION
# ============================================================

VOICE_PRESETS = {
    'female_us': 'en-US-AriaNeural',
    'male_us': 'en-US-GuyNeural',
    'female_uk': 'en-GB-SoniaNeural',
    'male_uk': 'en-GB-RyanNeural',
    'female_au': 'en-AU-NatashaNeural',
    'male_au': 'en-AU-WilliamNeural',
    'documentary': 'en-US-GuyNeural',
    'dramatic': 'en-GB-RyanNeural',
    'casual': 'en-US-AriaNeural',
    'energetic': 'en-US-JennyNeural',
}

# ⬇️ CHOOSE YOUR VOICE HERE ⬇️
CHOSEN_VOICE = 'documentary'  # Options: female_us, male_us, female_uk, male_uk, documentary, dramatic, casual, energetic

voice_name = VOICE_PRESETS[CHOSEN_VOICE]
print(f'🎙️ Voice: {CHOSEN_VOICE} ({voice_name})')
print(f'\nAvailable presets:')
for name, voice in VOICE_PRESETS.items():
    marker = ' ← selected' if name == CHOSEN_VOICE else ''
    print(f'   {name}: {voice}{marker}')

In [ ]:
# ============================================================
# GENERATE NARRATION AUDIO
# ============================================================

import edge_tts

async def generate_tts(text, output_path, voice, rate='+0%'):
    communicate = edge_tts.Communicate(text, voice, rate=rate)
    await communicate.save(str(output_path))

def get_audio_duration(path):
    try:
        result = subprocess.run(
            ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', str(path)],
            capture_output=True, text=True, timeout=10
        )
        return float(result.stdout.strip())
    except:
        return 5.0

print(f'🎙️ Generating narration for {len(narrations)} sections...')
print(f'   Voice: {voice_name}')
print('=' * 60)

audio_files = []
scene_num = 2  # Scene 1 is title card

for i, n in enumerate(narrations):
    output_path = AUDIO_DIR / f'narration_{scene_num:03d}.mp3'
    print(f'\n   Scene {scene_num}: Ch{n["chapter"]}.{n["section"]} [{n["mood"]}]')
    print(f'   Text: {n["narration"][:80]}...')
    print(f'   Generating...', end=' ', flush=True)

    try:
        await generate_tts(n['narration'], output_path, voice_name)
        duration = get_audio_duration(output_path)
        audio_files.append({
            'scene': scene_num,
            'path': str(output_path),
            'duration': duration,
            'text': n['narration']
        })
        print(f'✅ ({duration:.1f}s)')
    except Exception as e:
        print(f'❌ Error: {e}')

    scene_num += 1

# Save manifest
audio_manifest = {'voice': voice_name, 'files': audio_files}
with open(WORK_DIR / 'audio_manifest.json', 'w') as f:
    json.dump(audio_manifest, f, indent=2)

total_duration = sum(a['duration'] for a in audio_files)
print(f'\n{"=" * 60}')
print(f'✅ Generated {len(audio_files)} audio files ({total_duration:.1f}s total)')
print(f'📁 Saved to: {AUDIO_DIR}')

---
## 🔊 Preview Narration

Listen to each scene's narration:

In [ ]:
# PREVIEW AUDIO
for a in audio_files:
    display(Markdown(f"### Scene {a['scene']} ({a['duration']:.1f}s)"))
    display(Markdown(f"> {a['text'][:150]}..."))
    if Path(a['path']).exists():
        display(Audio(a['path'], autoplay=False))
    display(Markdown('---'))

print(f'\n💡 DISCUSSION:')
print(f'   Does this voice sound trustworthy? Authoritative?')
print(f'   Try changing CHOSEN_VOICE and re-running to compare.')
print(f'   How does the voice affect the meaning of the words?')

---
## ⏭️ Next Steps

Narration audio is on Google Drive. Open **06_Assemble.ipynb** to combine videos + audio into the final video.

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*